# Hybrid Anomaly Detection PoC

## Project Overview
**Title:** Proof of Concept (POC): Hybrid Anomaly-Based Fraud Detection for QR-Based Digital Transactions

**Objective:** Detect anomalous transactions in digital payment data using hybrid unsupervised ML (Isolation Forest + Autoencoder), explain predictions with SHAP (global map) and LIME (street-level detail), and generate natural language reasoning via AWS Bedrock.

**Technical Stack:**
- Python 3.9+
- scikit-learn (Isolation Forest)
- TensorFlow/Keras (Autoencoder)
- SHAP (KernelExplainer targeting Hybrid_Pipeline)
- LIME (LimeTabularExplainer)
- boto3 (AWS Bedrock Runtime API)
- pandas, numpy, matplotlib, seaborn

**Pipeline Phases:**
1. **Data Ingestion** — Synthetic generation or external CSV loading
2. **Preprocessing** — Missing value handling, feature engineering, scaling, splitting
3. **Anomaly Detection** — Isolation Forest + Autoencoder → Hybrid Score
4. **Explainability & LLM** — SHAP (map), LIME (detail), Bedrock (narrative)

## Dependencies

Required packages (minimum versions):
- scikit-learn>=1.3
- tensorflow>=2.13
- shap>=0.43
- lime>=0.2
- boto3>=1.34
- pandas>=2.0
- numpy>=1.24
- matplotlib>=3.7
- seaborn>=0.12

In [ ]:
# Uncomment to install dependencies
# !pip install scikit-learn>=1.3 tensorflow>=2.13 shap>=0.43 lime>=0.2 boto3>=1.34 pandas>=2.0 numpy>=1.24 matplotlib>=3.7 seaborn>=0.12

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import random
from pathlib import Path
from dataclasses import dataclass
from typing import NamedTuple, Callable, List, Tuple, Dict
from datetime import datetime, timedelta
import uuid
import json

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping

import shap
import lime
import lime.lime_tabular

import boto3
from botocore.exceptions import ClientError, NoCredentialsError

print('All imports successful!')

ModuleNotFoundError: No module named 'seaborn'

In [ ]:
@dataclass
class Config:
    '''Configuration for the hybrid anomaly detection POC.'''
    # Data Ingestion
    data_mode: str = 'synthetic'  # 'synthetic' or 'external'
    external_dataset_path: str = './data/external_transactions.csv'
    
    # Simulation (synthetic mode only)
    n_transactions: int = 1000
    anomaly_rate: float = 0.05
    random_seed: int = 42
    
    # Preprocessing
    test_size: float = 0.20
    
    # Isolation Forest
    if_contamination: float = 0.05
    if_n_estimators: int = 100
    
    # Autoencoder
    ae_epochs: int = 50
    ae_batch_size: int = 32
    ae_patience: int = 10
    
    # Hybrid Scoring
    w_if: float = 0.5
    w_ae: float = 0.5
    hybrid_threshold: float = 0.5
    
    # Explainability
    shap_top_n: int = 3
    lime_top_n: int = 3
    shap_background_size: int = 100
    
    # LLM
    bedrock_model_id: str = 'anthropic.claude-3-sonnet-20240229-v1:0'
    llm_top_k: int = 5

cfg = Config()
print(f'Config initialized: data_mode={cfg.data_mode}, n_transactions={cfg.n_transactions}')

In [ ]:
# Set global random seeds for reproducibility
np.random.seed(cfg.random_seed)
random.seed(cfg.random_seed)
tf.random.set_seed(cfg.random_seed)
print(f'Random seeds set to {cfg.random_seed}')

# Section 1: Data Ingestion

**Purpose:** Load or generate transaction dataset based on data_mode (synthetic or external).

**Inputs:** Config parameters (data_mode, external_dataset_path, n_transactions, anomaly_rate)

**Outputs:** raw_df (pd.DataFrame) with transaction records

In [ ]:
def generate_transactions(n: int, anomaly_rate: float, random_seed: int) -> pd.DataFrame:
    '''Generate synthetic transaction dataset.'''
    np.random.seed(random_seed)
    random.seed(random_seed)
    
    n_anomalies = int(n * anomaly_rate)
    n_normal = n - n_anomalies
    
    transactions = []
    
    # Normal transactions
    for i in range(n_normal):
        amount = np.random.lognormal(mean=4.5, sigma=0.8)
        frequency = np.random.poisson(lam=3)
        transactions.append({
            'transaction_id': str(uuid.uuid4()),
            'timestamp': datetime.utcnow() - timedelta(days=np.random.randint(0, 30)),
            'amount': amount,
            'merchant_id': f"M{np.random.randint(1, 21):03d}",
            'merchant_category': np.random.choice(['food', 'electronics', 'travel', 'retail', 'transport']),
            'device_id': f"D{np.random.randint(1, 51):03d}",
            'location': np.random.choice(['Jakarta', 'Surabaya', 'Bali', 'Bandung']),
            'payment_channel': np.random.choice(['QR', 'NFC', 'online']),
            'frequency': frequency,
            'is_anomaly': 0
        })
    
    # Anomalous transactions
    for i in range(n_anomalies):
        if np.random.rand() > 0.5:
            amount = np.random.uniform(500, 5000)
        else:
            amount = np.random.lognormal(mean=4.5, sigma=0.8)
        
        if np.random.rand() > 0.5:
            frequency = np.random.poisson(lam=20)
        else:
            frequency = np.random.poisson(lam=3)
        
        transactions.append({
            'transaction_id': str(uuid.uuid4()),
            'timestamp': datetime.utcnow() - timedelta(days=np.random.randint(0, 30)),
            'amount': amount,
            'merchant_id': f"M{np.random.randint(1, 21):03d}",
            'merchant_category': np.random.choice(['food', 'electronics', 'travel', 'retail', 'transport']),
            'device_id': f"D{np.random.randint(1, 51):03d}",
            'location': np.random.choice(['Jakarta', 'Surabaya', 'Bali', 'Bandung']),
            'payment_channel': np.random.choice(['QR', 'NFC', 'online']),
            'frequency': frequency,
            'is_anomaly': 1
        })
    
    df = pd.DataFrame(transactions)
    df = df.sample(frac=1, random_state=random_seed).reset_index(drop=True)
    return df

def _validate_columns(df: pd.DataFrame) -> None:
    '''Validate that required columns exist in external dataset.'''
    required = ['amount', 'frequency', 'device_id', 'location', 'payment_channel', 'merchant_category']
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {', '.join(missing)}")

def load_or_generate(cfg: Config) -> pd.DataFrame:
    '''Load or generate transaction dataset based on config.'''
    Path('./data').mkdir(exist_ok=True)
    
    if cfg.data_mode == 'synthetic':
        csv_path = Path('./data/transactions.csv')
        if csv_path.exists():
            print(f'Loading cached synthetic dataset from {csv_path}')
            return pd.read_csv(csv_path)
        else:
            print(f'Generating {cfg.n_transactions} synthetic transactions...')
            df = generate_transactions(cfg.n_transactions, cfg.anomaly_rate, cfg.random_seed)
            df.to_csv(csv_path, index=False)
            print(f'Saved to {csv_path}')
            return df
    elif cfg.data_mode == 'external':
        csv_path = Path(cfg.external_dataset_path)
        if not csv_path.exists():
            raise FileNotFoundError(f"External dataset not found: {cfg.external_dataset_path}")
        print(f'Loading external dataset from {csv_path}')
        df = pd.read_csv(csv_path)
        _validate_columns(df)
        return df
    else:
        raise ValueError(f"Unknown data_mode: {cfg.data_mode}")

# Load or generate data
raw_df = load_or_generate(cfg)
print(f'Dataset shape: {raw_df.shape}')
print(f'Columns: {list(raw_df.columns)}')

# Section 2: Preprocessing

**Purpose:** Clean, engineer features, encode categoricals, scale, and split data.

**Inputs:** raw_df (pd.DataFrame)

**Outputs:** PreprocessedData (X_train, X_test, y_train, y_test, scaler, feature_names)

In [ ]:
class PreprocessedData(NamedTuple):
    '''Container for preprocessed data.'''
    X_train: np.ndarray
    X_test: np.ndarray
    y_train: np.ndarray
    y_test: np.ndarray
    scaler: StandardScaler
    feature_names: List[str]

def preprocess(df: pd.DataFrame, cfg: Config) -> PreprocessedData:
    '''Preprocess transaction data.'''
    df = df.copy()
    
    # Drop rows with missing values in required columns
    required_cols = ['amount', 'frequency', 'device_id', 'location', 'payment_channel', 'merchant_category']
    initial_rows = len(df)
    df = df.dropna(subset=required_cols)
    dropped = initial_rows - len(df)
    if dropped > 0:
        print(f'Dropped {dropped} rows with missing values')
    
    if len(df) == 0:
        raise ValueError('Dataset empty after cleaning')
    
    # Feature engineering from timestamp
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df['hour_of_day'] = df['timestamp'].dt.hour
        df['day_of_week'] = df['timestamp'].dt.dayofweek
    
    # Prepare features and target
    y = df['is_anomaly'].values if 'is_anomaly' in df.columns else np.zeros(len(df))
    
    # Select features for encoding
    categorical_cols = ['merchant_category', 'device_id', 'location', 'payment_channel']
    if 'merchant_id' in df.columns:
        categorical_cols.insert(0, 'merchant_id')
    
    numeric_cols = ['amount', 'frequency']
    if 'hour_of_day' in df.columns:
        numeric_cols.extend(['hour_of_day', 'day_of_week'])
    
    # One-hot encode categorical columns
    df_encoded = pd.get_dummies(df[categorical_cols], drop_first=True)
    df_numeric = df[numeric_cols]
    
    X = pd.concat([df_numeric, df_encoded], axis=1)
    feature_names = list(X.columns)
    X = X.values
    
    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=cfg.test_size, random_state=cfg.random_seed, stratify=y if len(np.unique(y)) > 1 else None
    )
    
    if len(X_train) < 10 or len(X_test) < 10:
        raise ValueError(f'Train/test split too small: train={len(X_train)}, test={len(X_test)}')
    
    # Scale features
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
    print(f'X_train shape: {X_train.shape}')
    print(f'X_test shape: {X_test.shape}')
    
    return PreprocessedData(X_train, X_test, y_train, y_test, scaler, feature_names)

# Preprocess data
prep_data = preprocess(raw_df, cfg)
X_train, X_test, y_train, y_test = prep_data.X_train, prep_data.X_test, prep_data.y_train, prep_data.y_test
scaler, feature_names = prep_data.scaler, prep_data.feature_names

# Section 3: Isolation Forest

**Purpose:** Train Isolation Forest model for anomaly detection.

**Inputs:** X_train, X_test, cfg

**Outputs:** fitted_if_model, if_scores

In [ ]:
def train_isolation_forest(X_train: np.ndarray, contamination: float, n_estimators: int, random_seed: int) -> Tuple[IsolationForest, np.ndarray]:
    '''Train Isolation Forest and compute scores on test set.'''
    model = IsolationForest(contamination=contamination, n_estimators=n_estimators, random_state=random_seed)
    model.fit(X_train)
    
    if_scores = model.score_samples(X_test)
    predictions = model.predict(X_test)
    n_anomalies = (predictions == -1).sum()
    print(f'Isolation Forest: {n_anomalies} anomalies detected in test set')
    
    return model, if_scores

# Train Isolation Forest
if_model, if_scores = train_isolation_forest(X_train, cfg.if_contamination, cfg.if_n_estimators, cfg.random_seed)
print(f'IF scores shape: {if_scores.shape}')

# Section 4: Autoencoder

**Purpose:** Train Autoencoder on normal transactions for reconstruction error-based anomaly detection.

**Inputs:** X_train, y_train, X_test, cfg

**Outputs:** fitted_autoencoder, ae_scores, training loss plot

In [ ]:
class Autoencoder:
    '''Autoencoder for anomaly detection via reconstruction error.'''
    
    def __init__(self, input_dim: int, cfg: Config):
        self.input_dim = input_dim
        self.cfg = cfg
        self.model = self._build_model()
    
    def _build_model(self) -> Model:
        '''Build encoder-decoder architecture.'''
        D = self.input_dim
        
        inputs = keras.Input(shape=(D,))
        x = layers.Dense(D // 2, activation='relu')(inputs)
        x = layers.Dense(D // 4, activation='relu')(x)
        x = layers.Dense(max(D // 8, 4), activation='relu')(x)
        x = layers.Dense(D // 4, activation='relu')(x)
        x = layers.Dense(D // 2, activation='relu')(x)
        outputs = layers.Dense(D, activation='linear')(x)
        
        model = Model(inputs, outputs)
        model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss='mse')
        return model
    
    def fit(self, X_normal: np.ndarray) -> keras.callbacks.History:
        '''Train autoencoder on normal samples.'''
        early_stop = EarlyStopping(monitor='val_loss', patience=self.cfg.ae_patience, restore_best_weights=True)
        history = self.model.fit(
            X_normal, X_normal,
            epochs=self.cfg.ae_epochs,
            batch_size=self.cfg.ae_batch_size,
            validation_split=0.1,
            callbacks=[early_stop],
            verbose=0
        )
        return history
    
    def reconstruction_error(self, X: np.ndarray) -> np.ndarray:
        '''Compute per-sample reconstruction error (MSE).'''
        X_recon = self.model.predict(X, verbose=0)
        mse = np.mean((X - X_recon) ** 2, axis=1)
        return mse

# Train Autoencoder on normal samples only
X_train_normal = X_train[y_train == 0]
if len(X_train_normal) == 0:
    raise ValueError('No normal samples for Autoencoder training')

autoencoder = Autoencoder(X_train.shape[1], cfg)
history = autoencoder.fit(X_train_normal)
ae_scores = autoencoder.reconstruction_error(X_test)
print(f'AE scores shape: {ae_scores.shape}')

# Plot training history
plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.title('Autoencoder Training History')
plt.grid(True, alpha=0.3)
plt.show()

# Section 5: Hybrid Scoring

**Purpose:** Combine IF and AE scores into hybrid anomaly score and classify transactions.

**Inputs:** if_scores, ae_scores, X_test, y_test, feature_names, cfg

**Outputs:** AnomalyResult list, classification report, confusion matrix plot

In [ ]:
@dataclass
class AnomalyResult:
    '''Result of hybrid anomaly detection for a single transaction.'''
    transaction_id: str
    if_score_raw: float
    ae_score_raw: float
    if_score_norm: float
    ae_score_norm: float
    hybrid_score: float
    predicted_label: int
    true_label: int
    features: Dict

def compute_hybrid_scores(if_scores: np.ndarray, ae_scores: np.ndarray, w_if: float = 0.5, w_ae: float = 0.5, threshold: float = 0.5) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    '''Compute hybrid anomaly scores.'''
    if abs(w_if + w_ae - 1.0) > 1e-9:
        raise ValueError(f'Weights must sum to 1.0, got {w_if + w_ae}')
    
    # Invert IF scores (more negative = more anomalous, so invert to make higher = more anomalous)
    if_inv = -if_scores
    
    # Min-max normalize
    def minmax_norm(arr):
        min_val, max_val = arr.min(), arr.max()
        if max_val == min_val:
            print(f'Warning: constant scores detected, setting normalized values to 0.0')
            return np.zeros_like(arr)
        return (arr - min_val) / (max_val - min_val)
    
    if_norm = minmax_norm(if_inv)
    ae_norm = minmax_norm(ae_scores)
    
    # Weighted average
    hybrid = w_if * if_norm + w_ae * ae_norm
    
    # Classify
    predicted = (hybrid >= threshold).astype(int)
    
    return hybrid, predicted, if_norm, ae_norm

# Compute hybrid scores
hybrid_scores, y_pred, if_norm, ae_norm = compute_hybrid_scores(if_scores, ae_scores, cfg.w_if, cfg.w_ae, cfg.hybrid_threshold)

# Create AnomalyResult objects
anomaly_results = []
for i in range(len(X_test)):
    result = AnomalyResult(
        transaction_id=f'txn_{i}',
        if_score_raw=if_scores[i],
        ae_score_raw=ae_scores[i],
        if_score_norm=if_norm[i],
        ae_score_norm=ae_norm[i],
        hybrid_score=hybrid_scores[i],
        predicted_label=y_pred[i],
        true_label=int(y_test[i]),
        features={}
    )
    anomaly_results.append(result)

# Classification report
print('\n=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['Normal', 'Anomaly']))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Normal', 'Anomaly'], yticklabels=['Normal', 'Anomaly'])
plt.title('Confusion Matrix - Hybrid Anomaly Detector')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

In [ ]:
def hybrid_predict_fn(X: np.ndarray) -> np.ndarray:
    '''Wrapper for hybrid pipeline - returns probabilities for SHAP/LIME.'''
    if_scores_local = if_model.score_samples(X)
    ae_scores_local = autoencoder.reconstruction_error(X)
    
    hybrid_local, _, if_norm_local, ae_norm_local = compute_hybrid_scores(
        if_scores_local, ae_scores_local, cfg.w_if, cfg.w_ae, cfg.hybrid_threshold
    )
    
    # Return [P(normal), P(anomaly)]
    proba = np.column_stack([1 - hybrid_local, hybrid_local])
    return proba

print('hybrid_predict_fn defined')

# Section 6: SHAP Explainability

**Purpose:** Explain hybrid anomaly scores using SHAP (global map + local detail).

**Inputs:** hybrid_predict_fn, X_train, X_test, feature_names, cfg

**Outputs:** SHAPExplanation list, global summary plot, local waterfall plots

In [ ]:
@dataclass
class SHAPExplanation:
    '''SHAP explanation for a transaction.'''
    transaction_id: str
    shap_values: np.ndarray
    top_features: List[Tuple[str, float]]
    base_value: float

def explain_shap(hybrid_predict_fn: Callable, X_train: np.ndarray, X_test: np.ndarray, feature_names: List[str], top_n: int = 3) -> List[SHAPExplanation]:
    '''Compute SHAP explanations for test set.'''
    print('Computing SHAP values (this may take a minute)...')
    
    # Create background sample
    background_indices = np.random.choice(len(X_train), size=min(100, len(X_train)), replace=False)
    background = X_train[background_indices]
    
    # Create explainer
    explainer = shap.KernelExplainer(hybrid_predict_fn, background)
    
    # Compute SHAP values for test set
    shap_values = explainer.shap_values(X_test)
    
    # Global summary plot
    print('\nGlobal SHAP Summary (Feature Importance):')
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values[1], X_test, feature_names=feature_names, plot_type='bar', show=False)
    plt.tight_layout()
    plt.show()
    
    # Create explanations for anomalies
    explanations = []
    anomaly_indices = np.where(y_pred == 1)[0]
    
    for idx in anomaly_indices[:min(3, len(anomaly_indices))]:
        try:
            # Local waterfall plot
            plt.figure(figsize=(10, 6))
            shap.waterfall_plot(shap.Explanation(
                values=shap_values[1][idx],
                base_values=explainer.expected_value[1],
                data=X_test[idx],
                feature_names=feature_names
            ), show=False)
            plt.tight_layout()
            plt.show()
            
            # Extract top features
            top_indices = np.argsort(np.abs(shap_values[1][idx]))[-top_n:][::-1]
            top_features = [(feature_names[i], shap_values[1][idx][i]) for i in top_indices]
            
            explanations.append(SHAPExplanation(
                transaction_id=f'txn_{idx}',
                shap_values=shap_values[1][idx],
                top_features=top_features,
                base_value=explainer.expected_value[1]
            ))
        except Exception as e:
            print(f'Error computing SHAP for sample {idx}: {e}')
            explanations.append(SHAPExplanation(
                transaction_id=f'txn_{idx}',
                shap_values=np.zeros(len(feature_names)),
                top_features=[],
                base_value=0.0
            ))
    
    return explanations

# Compute SHAP explanations
shap_explanations = explain_shap(hybrid_predict_fn, X_train, X_test, feature_names, cfg.shap_top_n)
print(f'\nComputed {len(shap_explanations)} SHAP explanations')

# Section 7: LIME Explainability

**Purpose:** Explain individual anomalous transactions using LIME (street-level detail).

**Inputs:** hybrid_predict_fn, X_train, X_test, feature_names, cfg

**Outputs:** LIMEExplanation list, local bar charts

In [ ]:
@dataclass
class LIMEExplanation:
    '''LIME explanation for a transaction.'''
    transaction_id: str
    top_features: List[Tuple[str, float]]
    local_pred: float

def explain_lime(hybrid_predict_fn: Callable, X_train: np.ndarray, X_test: np.ndarray, feature_names: List[str], top_n: int = 3) -> List[LIMEExplanation]:
    '''Compute LIME explanations for anomalous transactions.'''
    print('Computing LIME explanations...')
    
    explainer = lime.lime_tabular.LimeTabularExplainer(
        X_train, feature_names=feature_names, mode='classification', verbose=False
    )
    
    explanations = []
    anomaly_indices = np.where(y_pred == 1)[0]
    
    for idx in anomaly_indices[:min(3, len(anomaly_indices))]:
        try:
            exp = explainer.explain_instance(X_test[idx], hybrid_predict_fn, num_features=top_n)
            
            # Render bar chart
            fig = exp.as_pyplot_figure()
            plt.tight_layout()
            plt.show()
            
            # Extract top features
            top_features = [(name, weight) for name, weight in exp.as_list()]
            local_pred = exp.predict_proba[1]  # P(anomaly)
            
            explanations.append(LIMEExplanation(
                transaction_id=f'txn_{idx}',
                top_features=top_features,
                local_pred=local_pred
            ))
        except Exception as e:
            print(f'Error computing LIME for sample {idx}: {e}')
            explanations.append(LIMEExplanation(
                transaction_id=f'txn_{idx}',
                top_features=[],
                local_pred=0.0
            ))
    
    return explanations

# Compute LIME explanations
lime_explanations = explain_lime(hybrid_predict_fn, X_train, X_test, feature_names, cfg.lime_top_n)
print(f'\nComputed {len(lime_explanations)} LIME explanations')

# Section 8: LLM Reasoning

**Purpose:** Generate natural language explanations using AWS Bedrock.

**Inputs:** anomaly_results, shap_explanations, lime_explanations, cfg

**Outputs:** ExplanationRecord list, LLM narratives (inline)

In [ ]:
@dataclass
class ExplanationRecord:
    '''Complete explanation record for a transaction.'''
    transaction_id: str
    anomaly_result: AnomalyResult
    shap_explanation: SHAPExplanation
    lime_explanation: LIMEExplanation
    llm_narrative: str
    prompt_sent: str

def build_prompt(anomaly_result: AnomalyResult, shap_exp: SHAPExplanation, lime_exp: LIMEExplanation, cfg: Config) -> str:
    '''Build prompt for LLM reasoning.'''
    shap_features = '\n'.join([f'- {name}: {value:+.4f}' for name, value in shap_exp.top_features[:cfg.shap_top_n]])
    lime_features = '\n'.join([f'- {name}: {value:+.4f}' for name, value in lime_exp.top_features[:cfg.lime_top_n]])
    
    prompt = f'''You are a financial fraud analyst reviewing QR-payment transactions.
A hybrid anomaly detection system (Isolation Forest + Autoencoder) has flagged the following transaction as suspicious.

=== Transaction Details ===
Transaction ID: {anomaly_result.transaction_id}
Hybrid Anomaly Score: {anomaly_result.hybrid_score:.4f}

=== Anomaly Scores ===
Isolation Forest (normalized): {anomaly_result.if_score_norm:.4f}
Autoencoder reconstruction error (normalized): {anomaly_result.ae_score_norm:.4f}

=== SHAP Top Contributing Features ===
{shap_features}

=== LIME Top Contributing Features ===
{lime_features}

=== Task ===
1. Explain in plain English why this transaction is suspicious.
2. Identify the single most important risk factor.
3. Suggest one concrete follow-up action.

Keep your response concise (≤ 150 words).'''
    
    return prompt

def explain_with_llm(anomaly_results: List[AnomalyResult], shap_exps: List[SHAPExplanation], lime_exps: List[LIMEExplanation], cfg: Config) -> List[ExplanationRecord]:
    '''Generate LLM explanations for top-K anomalies.'''
    # Sort by hybrid score and select top-K
    sorted_indices = np.argsort([r.hybrid_score for r in anomaly_results])[::-1]
    top_indices = sorted_indices[:min(cfg.llm_top_k, len(anomaly_results))]
    
    records = []
    
    for idx in top_indices:
        anomaly_result = anomaly_results[idx]
        shap_exp = shap_exps[min(len(shap_exps)-1, idx)] if shap_exps else SHAPExplanation('', np.array([]), [], 0.0)
        lime_exp = lime_exps[min(len(lime_exps)-1, idx)] if lime_exps else LIMEExplanation('', [], 0.0)
        
        prompt = build_prompt(anomaly_result, shap_exp, lime_exp, cfg)
        
        # Try to call Bedrock
        llm_narrative = 'LLM unavailable: AWS credentials not configured'
        try:
            region = os.environ.get('AWS_DEFAULT_REGION', 'us-east-1')
            client = boto3.client('bedrock-runtime', region_name=region)
            
            response = client.invoke_model(
                modelId=cfg.bedrock_model_id,
                body=json.dumps({
                    'prompt': prompt,
                    'max_tokens_to_sample': 300,
                    'temperature': 0.7
                })
            )
            
            response_body = json.loads(response['body'].read())
            llm_narrative = response_body.get('completion', 'LLM unavailable: response parse error')
        except NoCredentialsError:
            llm_narrative = 'LLM unavailable: AWS credentials not configured'
        except ClientError as e:
            if 'ThrottlingException' in str(e) or 'ServiceUnavailableException' in str(e):
                llm_narrative = f'LLM unavailable: {str(e)[:50]}'
            else:
                llm_narrative = f'LLM unavailable: {str(e)[:50]}'
        except Exception as e:
            llm_narrative = f'LLM unavailable: {str(e)[:50]}'
        
        records.append(ExplanationRecord(
            transaction_id=anomaly_result.transaction_id,
            anomaly_result=anomaly_result,
            shap_explanation=shap_exp,
            lime_explanation=lime_exp,
            llm_narrative=llm_narrative,
            prompt_sent=prompt
        ))
        
        # Display inline
        print(f'\n=== {anomaly_result.transaction_id} ===')
        print(f'Hybrid Score: {anomaly_result.hybrid_score:.4f}')
        print(f'\nLLM Narrative:\n{llm_narrative}')
    
    return records

# Generate LLM explanations
explanation_records = explain_with_llm(anomaly_results, shap_explanations, lime_explanations, cfg)
print(f'\nGenerated {len(explanation_records)} explanation records')

# Section 9: Results Summary

**Purpose:** Consolidate and display all results in a single summary table.

**Inputs:** explanation_records

**Outputs:** Summary DataFrame with all key metrics and narratives

In [ ]:
# Create results summary
summary_data = []
for record in explanation_records:
    summary_data.append({
        'Transaction ID': record.transaction_id,
        'Hybrid Score': f"{record.anomaly_result.hybrid_score:.4f}",
        'IF Score (norm)': f"{record.anomaly_result.if_score_norm:.4f}",
        'AE Score (norm)': f"{record.anomaly_result.ae_score_norm:.4f}",
        'Predicted': 'Anomaly' if record.anomaly_result.predicted_label == 1 else 'Normal',
        'True Label': 'Anomaly' if record.anomaly_result.true_label == 1 else 'Normal',
        'LLM Narrative': record.llm_narrative[:100] + '...' if len(record.llm_narrative) > 100 else record.llm_narrative
    })

summary_df = pd.DataFrame(summary_data)
print('\n=== Results Summary ===')
print(summary_df.to_string(index=False))
print(f'\nTotal anomalies detected: {(y_pred == 1).sum()}')
print(f'Total transactions: {len(y_pred)}')

In [ ]:
# Verify all outputs are inline
print('\n=== Verification Checklist ===')
print('✓ Data pipeline: Loaded/generated transactions')
print('✓ Preprocessing: Scaled and split data')
print('✓ Isolation Forest: Trained and scored')
print('✓ Autoencoder: Trained with loss curve plot')
print('✓ Hybrid Scoring: Computed scores and confusion matrix')
print('✓ SHAP: Global summary and local waterfall plots')
print('✓ LIME: Local bar charts for anomalies')
print('✓ LLM: Generated narratives (or fallback messages)')
print('✓ Results: Summary table displayed')
print('\nAll outputs rendered inline in notebook cells!')